# 06 — Multivariate Composition
**Goal:** Show many variables at once without overwhelming the reader —
bubble, parallel coordinates, radar, heatmap, small multiples.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

np.random.seed(21)
print('numpy', np.__version__)

## 1. The four-or-more-variable problem

A scatter can show ~3 variables (x, y, color). Anything beyond that needs a
*different* geometry:

| Variables | Best chart |
|---|---|
| 2 (continuous) | scatter, line, bar |
| 3 (2 cont + 1 cat) | colored scatter / bar |
| 4 (3 cont + 1 cat) | bubble |
| 5+ (many cont + cat) | parallel coordinates, radar, heatmap, small multiples |
| Matrix (rows × cols) | heatmap |

Rule: if you have to draw a legend with 12 entries, redesign.

## 2. Bubble chart with 4 variables

Two continuous on x/y, a category on color, a continuous on size. Mind the
overplotting.

In [ ]:
n = 60
country = np.random.choice(['US', 'UK', 'DE', 'JP', 'BR'], n)
gdp     = np.random.uniform(10, 60, n)
lifeexp = 60 + 0.5 * gdp + np.random.normal(0, 4, n)
pop     = np.random.uniform(1, 100, n)
colors  = {'US': 'steelblue', 'UK': 'crimson', 'DE': 'seagreen', 'JP': 'orange', 'BR': 'purple'}

fig, ax = plt.subplots(figsize=(8, 5))
for c, color in colors.items():
    mask = country == c
    ax.scatter(gdp[mask], lifeexp[mask], s=pop[mask] * 8,
               alpha=0.55, color=color, label=c, edgecolor='white')
ax.set_xlabel('GDP per capita (k$)'); ax.set_ylabel('Life expectancy')
ax.set_title('Bubble — x, y, color, size')
ax.legend(frameon=False, title='country', loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 3. Heatmap — the matrix chart

A heatmap shows a 2D matrix where color encodes value. The most common form
in growth analytics is *cohort retention*: rows are cohorts, columns are
months since acquisition, color is retention %.

In [ ]:
cohorts = pd.date_range('2024-01-01', periods=8, freq='MS')
M = 12
mat = np.zeros((len(cohorts), M))
for i in range(len(cohorts)):
    for j in range(M - i):
        mat[i, j] = 1.0 * np.exp(-0.15 * j) + np.random.normal(0, 0.02)
mat = np.clip(mat, 0, 1)
mat[np.tril_indices_from(mat, k=1)] = np.nan

fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(mat, aspect='auto', cmap='YlGnBu', vmin=0, vmax=1)
ax.set_yticks(range(len(cohorts)))
ax.set_yticklabels([c.strftime('%Y-%m') for c in cohorts])
ax.set_xticks(range(M))
ax.set_xticklabels([f'M+{j}' for j in range(M)])
ax.set_title('Cohort retention heatmap')
fig.colorbar(im, ax=ax, label='retention')
plt.tight_layout(); plt.show()

## 4. Correlation heatmap

Use when you have a wide table and want to surface *which variables move
together*. Always annotate with the actual value.

In [ ]:
df = pd.DataFrame(np.random.randn(200, 5),
                  columns=['spend', 'reach', 'ctr', 'conv', 'rev'])
df['rev'] = df['rev'] + 0.6 * df['spend']
df['conv'] = df['conv'] + 0.4 * df['ctr']
corr = df.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                color='white' if abs(corr.iloc[i, j]) > 0.6 else 'black', fontsize=9)
fig.colorbar(im, ax=ax, label='correlation')
ax.set_title('Correlation heatmap')
plt.tight_layout(); plt.show()

## 5. Parallel coordinates

Each variable gets a vertical axis. Each observation is a line that crosses
all of them. Use it to find clusters and outliers across many dimensions.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=120, centers=3, n_features=4, random_state=0)
X = MinMaxScaler().fit_transform(X)
labels = [f'cluster {c}' for c in y]
cols   = ['spend', 'reach', 'ctr', 'conv']
cmap   = plt.cm.tab10

fig, ax = plt.subplots(figsize=(9, 4.5))
for j in range(X.shape[1]):
    ax.plot([j, j], [0, 1], 'k-', lw=0.5)
for i in range(X.shape[0]):
    ax.plot(range(X.shape[1]), X[i], color=cmap(y[i] % 10), alpha=0.5, lw=0.8)
ax.set_xticks(range(X.shape[1])); ax.set_xticklabels(cols)
ax.set_ylim(-0.05, 1.05)
ax.set_title('Parallel coordinates — 3 clusters across 4 variables')
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color=cmap(c % 10), lw=2,
                  label=f'cluster {c}') for c in sorted(set(y))]
ax.legend(handles=handles, frameon=False, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Radar / spider chart — when (and when not) to use it

Radar plots normalize each axis to a common range and draw a polygon per
observation. They are *visually attractive* and *statistically misleading*:
the area scales non-linearly with the values, and the order of axes changes
the perceived shape.

Use them only for **profile comparison across a small, fixed set of
dimensions** (e.g. brand attributes, KPI scores) where the polygon metaphor
matches the reader's mental model.

In [ ]:
categories = ['Speed', 'Reliability', 'Price', 'Support', 'UX']
productA   = [4, 5, 3, 4, 5]
productB   = [3, 3, 5, 4, 3]
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]
productA += productA[:1]
productB += productB[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles, productA, color='steelblue', lw=2, label='Product A')
ax.fill(angles, productA, color='steelblue', alpha=0.2)
ax.plot(angles, productB, color='crimson', lw=2, label='Product B')
ax.fill(angles, productB, color='crimson', alpha=0.2)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories)
ax.set_ylim(0, 5)
ax.set_title('Radar — profile comparison')
ax.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.show()

## 7. Small multiples — the safe default

When in doubt, repeat the same simple chart in a grid, one panel per category.
This is the *most scalable* approach to multivariate data.

We will go much deeper in notebook 09; this is just a taste.

In [ ]:
channels = ['Search', 'Social', 'Display', 'Email']
data = {c: np.random.normal(loc=np.random.uniform(-1, 1), scale=1, size=200) for c in channels}
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, ch in zip(axes.ravel(), channels):
    ax.hist(data[ch], bins=25, color='steelblue', edgecolor='white')
    ax.set_title(ch)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Small multiples — same chart, one per channel', x=0.05, ha='left', weight='bold')
plt.tight_layout(); plt.show()

## 8. Choosing a diverging vs sequential colormap

- **Sequential** (e.g. `viridis`, `YlGnBu`) for ordered data 0 → max.
- **Diverging** (e.g. `RdBu_r`, `BrBG`) for signed data around a meaningful
  center (e.g. correlation, % change vs baseline).
- **Qualitative** (e.g. `tab10`, `Set2`) for unordered categories.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 2.2))
for ax, cmap, name in zip(axes, ['viridis', 'RdBu_r', 'tab10'],
                          ['sequential', 'diverging', 'qualitative']):
    grad = np.linspace(0, 1, 256).reshape(1, -1)
    ax.imshow(grad, aspect='auto', cmap=cmap)
    ax.set_title(name); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Summary

| Chart | Variables | Watch out for |
|---|---|---|
| Bubble | 4 (2 cont + cat + cont) | Overplotting, area perception |
| Heatmap | matrix | Hidden row/column order effects |
| Correlation heatmap | relationships | Spurious correlations |
| Parallel coords | many cont | Cluttered with too many series |
| Radar | fixed profile | Misleading area, axis order |
| Small multiples | any | Inconsistent scales across panels |

**Next:** `07_seaborn_statistical.ipynb` — high-level statistical plotting.